## introdução

### resumo do enunciado (extraído do arquivo `atividade 2.pdf`)

1. **problema:** classificar o sentimento de comentários (reviews) sobre revistas como **positivo (`good`)** ou **negativo (`bad`)** (análise de sentimentos supervisionada).
2. **dados de entrada:** três arquivos — treino anotado, teste sem rótulos e exemplo de submissão (formato kaggle).
3. **saídas esperadas:** arquivo de submissão com predições para o conjunto de teste; o desempenho é avaliado automaticamente na competição.
4. **modelos obrigatórios (scikit-learn):** k vizinhos mais próximos (knn), naive bayes e árvore de decisão; uso de **pandas** para leitura em formato tabular.
5. **entregas obrigatórias:** desenvolvimento no notebook, submissão no link da competição; trabalho individual ou em grupo de até 3 pessoas.
6. **metas extras:** **0,5 ponto na primeira prova** se atingir **acurácia mínima de 87%** no desafio (medida **no kaggle**, não no holdout local deste notebook), publicar o **notebook completo e reproduzível** no fórum do moodle; apenas um integrante envia a submissão final no kaggle.

**escopo:** seguimos o enunciado: apenas **knn**, **naive bayes (multinomial)** e **árvore de decisão** com **pandas** e **scikit-learn**. a nota de **87%** da bonificação é computada **no kaggle**; neste notebook avaliamos só um holdout local no `train.csv`.

### decisão documentada sobre os arquivos

os csv usados estão na pasta `aprendizado-de-maquina-26-1-competicao-2-pln/` (no repositório não há subpasta `dados`; o conteúdo corresponde ao pedido: `train.csv`, `test.csv`, `sample_submission.csv`). o notebook assume esse caminho para tudo rodar de forma reproduzível a partir da raiz do projeto.

### o que o código base já fazia

o material de exemplo carregava `train.csv`/`test.csv`, fazia um `train_test_split`, usava `countvectorizer` + `multinomialnb` default, media acurácia no holdout e gerava `submission.csv` com a coluna **`target`**, que **não** coincide com o `sample_submission.csv` oficial (coluna **`label`**). abaixo reaproveitamos a ideia (pandas + vetorização + sklearn + função de exportação), corrigimos a submissão e acrescentamos os três modelos, comparações e pré-processamento.

## leitura dos dados

definimos o diretório dos arquivos, carregamos treino e teste e usamos `id` como índice, como no exemplo original.

In [ ]:
import re
import warnings
from itertools import product
from pathlib import Path

import pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
DATA_DIR = Path("aprendizado-de-maquina-26-1-competicao-2-pln")

train_df = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
test_df = pd.read_csv(DATA_DIR / "test.csv", index_col="id")

## exploração básica

In [ ]:
print(train_df.shape, test_df.shape)
print("nulos em review (treino):", train_df["review"].isna().sum())
print("rótulos:\n", train_df["label"].value_counts())
print("\nexemplo de review bruto:\n", train_df["review"].iloc[0][:400])

## pré-processamento

há tags html (`<p>`, `<br>`, etc.) e espaços irregulares. aplicamos limpeza **simples e reproduzível**: remover tags com expressão regular, converter para minúsculas e normalizar espaços. não removemos pontuação de propósito para não alterar demais o sinal dos unigramas/bigramas; `stop_words` e `min_df` no vetorizador já ajudam a filtrar ruído frequente.

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")


def limpar_texto(texto) -> str:
    if not isinstance(texto, str):
        texto = str(texto)
    texto = TAG_RE.sub(" ", texto)
    texto = texto.lower()
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


train_df["review_limpo"] = train_df["review"].map(limpar_texto)
test_df["review_limpo"] = test_df["review"].map(limpar_texto)

## vetorização

definimos **countvectorizer** e **tfidfvectorizer** com unigramas, unigramas+bigramas, `min_df`, `max_df` (quando aplicável) e `stop_words` em inglês (texto dos reviews está em inglês). na mesma célula de código, fazemos a **divisão 80% treino / 20% validação** estratificada só com `train.csv`, para calibrar hiperparâmetros sem usar o teste do kaggle. na seção **modelos testados**, avaliamos knn, multinomial naive bayes e árvore de decisão, como exige o enunciado.

In [ ]:
y = train_df["label"]
X = train_df["review_limpo"]

X_treino, X_val, y_treino, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

vectorizadores = [
    (
        "count_unigrama",
        CountVectorizer(
            ngram_range=(1, 1),
            min_df=2,
            max_df=0.95,
            stop_words="english",
        ),
    ),
    (
        "count_unigrama_bigrama",
        CountVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=20_000,
            stop_words="english",
        ),
    ),
    (
        "tfidf_unigrama",
        TfidfVectorizer(
            ngram_range=(1, 1),
            min_df=2,
            max_features=15_000,
            stop_words="english",
            sublinear_tf=True,
        ),
    ),
    (
        "tfidf_unigrama_bigrama",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=20_000,
            stop_words="english",
            sublinear_tf=True,
        ),
    ),
]

knn_grade = list(product([3, 5, 7, 9], ["uniform", "distance"], ["cosine"]))
nb_alphas = [0.01, 0.1, 0.5, 1.0]
arvore_grade = list(product([15, 25, 40], [2, 5], ["gini", "entropy"]))

## modelos testados

In [ ]:
resultados = []
melhor_acc = -1.0
melhor_pipeline = None

for nome_vet, vet_base in vectorizadores:
    for k, pesos, metrica in knn_grade:
        pipe = Pipeline(
            [
                ("vet", clone(vet_base)),
                (
                    "clf",
                    KNeighborsClassifier(
                        n_neighbors=k,
                        weights=pesos,
                        metric=metrica,
                        algorithm="brute",
                        n_jobs=-1,
                    ),
                ),
            ]
        )
        pipe.fit(X_treino, y_treino)
        acc = accuracy_score(y_val, pipe.predict(X_val))
        resultados.append(
            {
                "vetorizacao": nome_vet,
                "modelo": "kneighborsclassifier",
                "hiperparametros": f"n_neighbors={k}, weights={pesos}, metric={metrica}, algorithm=brute",
                "acuracia_validacao": acc,
            }
        )
        if acc > melhor_acc:
            melhor_acc = acc
            melhor_pipeline = clone(pipe)

    for alpha in nb_alphas:
        pipe = Pipeline(
            [("vet", clone(vet_base)), ("clf", MultinomialNB(alpha=alpha))]
        )
        pipe.fit(X_treino, y_treino)
        acc = accuracy_score(y_val, pipe.predict(X_val))
        resultados.append(
            {
                "vetorizacao": nome_vet,
                "modelo": "multinomial_nb",
                "hiperparametros": f"alpha={alpha}",
                "acuracia_validacao": acc,
            }
        )
        if acc > melhor_acc:
            melhor_acc = acc
            melhor_pipeline = clone(pipe)

    for profundidade, mss, crit in arvore_grade:
        pipe = Pipeline(
            [
                ("vet", clone(vet_base)),
                (
                    "clf",
                    DecisionTreeClassifier(
                        max_depth=profundidade,
                        min_samples_split=mss,
                        criterion=crit,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
        pipe.fit(X_treino, y_treino)
        acc = accuracy_score(y_val, pipe.predict(X_val))
        resultados.append(
            {
                "vetorizacao": nome_vet,
                "modelo": "decisiontreeclassifier",
                "hiperparametros": f"max_depth={profundidade}, min_samples_split={mss}, criterion={crit}, random_state={RANDOM_STATE}",
                "acuracia_validacao": acc,
            }
        )
        if acc > melhor_acc:
            melhor_acc = acc
            melhor_pipeline = clone(pipe)

## comparação de resultados

tabela ordenada por acurácia no conjunto de validação (holdout). cada linha é uma combinação: vetorização + modelo + hiperparâmetros.

In [ ]:
tabela = pd.DataFrame(resultados).sort_values(
    "acuracia_validacao", ascending=False
)
tabela.reset_index(drop=True, inplace=True)
pd.set_option("display.max_rows", 200)
tabela

## escolha do melhor modelo

selecionamos o pipeline com **maior acurácia na validação** (mesmo holdout `random_state=42`). o objeto `melhor_pipeline` é um **clone não treinado** da configuração vencedora; abaixo ele é **ajustado em todo o `train.csv`** antes das predições no teste.

**justificativa:** o holdout estratificado estima o erro de generalização com os rótulos disponíveis; escolher o melhor valor nessa métrica é o critério mais alinhado ao enunciado quando não há conjunto de validação externo.

In [10]:
print("melhor configuração (validação):\n", tabela.iloc[0].to_string())

pipeline_final = clone(melhor_pipeline)
pipeline_final.fit(X, y)

acuracia_validacao_melhor = float(tabela.iloc[0]["acuracia_validacao"])
print("acurácia de validação (melhor linha da tabela):", acuracia_validacao_melhor)

melhor configuração (validação):
 vetorizacao           count_unigrama
modelo                multinomial_nb
hiperparametros            alpha=0.1
acuracia_validacao          0.634933
acurácia de validação (melhor linha da tabela): 0.6349325337331334


## geração da submissão

In [11]:
def create_submission_file(predictions, test_df, submission_file_name="submission.csv"):
    submission_df = pd.DataFrame({"id": test_df.index, "label": predictions})
    submission_df.to_csv(submission_file_name, index=False)
    print(f"arquivo '{submission_file_name}' criado.")


predicoes_teste = pipeline_final.predict(test_df["review_limpo"])
create_submission_file(predicoes_teste, test_df, "submission.csv")


arquivo 'submission.csv' criado.


## conclusão

- foi aplicada **limpeza textual** mínima e comparadas representações **bag-of-words** e **tf-idf** com n-gramas e filtros de frequência.
- **knn**, **multinomial naive bayes** e **árvore de decisão** foram avaliados com grades de hiperparâmetros modestas para manter o notebook executável em tempo razoável.
- o **melhor modelo entre os três pedidos no pdf** está na tabela; a acurácia de validação correspondente é `acuracia_validacao_melhor`.
- a **meta de 87%** do enunciato refere-se ao **desempenho na competição (kaggle)**; o valor local não substitui essa métrica.

### referência ao código base original

abaixo, o trecho equivalente ao notebook de exemplo (imports e fluxo simples), útil para comparação com o baseline **sem** limpeza e com coluna de submissão incorreta — **não é necessário executar** para a entrega final.

In [12]:
# baseline de referência (código base) — não reexecutar no fluxo principal
# import pandas as pd
# from sklearn.metrics import accuracy_score
# from sklearn.feature_extraction.text import CountVectorizer
# from sklearn.naive_bayes import MultinomialNB
# from sklearn.model_selection import train_test_split
# train_df = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
# X_tr, X_te, y_tr, y_te = train_test_split(train_df["review"], train_df["label"], test_size=0.2, random_state=42)
# cv = CountVectorizer().fit(X_tr)
# modelo = MultinomialNB().fit(cv.transform(X_tr), y_tr)
# print(accuracy_score(y_te, modelo.predict(cv.transform(X_te))))